# log-back — worked example 1: log_back as elementwise grad_out / x

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `log-back`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The forward op is `out = log(x)`. Since `d/dx log(x) = 1/x` and the op is elementwise, the Jacobian is diagonal and the chain rule collapses to a per-position product: `grad_x = grad_out * (1/x) = grad_out / x`. No Jacobian is ever materialized.

## Worked solution

We implement the canonical first backward rule and check it against autograd.

1. **The rule.** `log_back(grad_out, out, x)` returns `grad_out / x`. The `out` argument is part of the uniform `(grad_out, out, *fwd_args)` signature every back fn shares so the dispatcher can call them generically; `log_back` simply does not read it.
2. **Why division.** The local derivative of `log` at position `i` is `1/x[i]`. Multiplying the upstream gradient by it is exactly elementwise division by `x`.
3. **Witness.** We build `x` with `requires_grad=True`, compute `loss = (log(x) * grad_out).sum()`, and call `loss.backward()`. The accumulated `x.grad` must equal our hand-rolled `log_back` output.

The demo prints the hand-rolled gradient next to autograd's and confirms they match.

In [ ]:
import torch as t

t.manual_seed(0)

def log_back(grad_out, out, x):
    # d/dx log(x) = 1/x; chain rule => grad_out / x. `out` unused.
    return grad_out / x

x = t.rand(5) + 0.5            # strictly positive
grad_out = t.randn(5)
out = t.log(x)
hand = log_back(grad_out, out, x)

x_var = x.clone().detach().requires_grad_(True)
(t.log(x_var) * grad_out).sum().backward()
print('hand:', hand)
print('autograd matches:', t.allclose(hand, x_var.grad))